# **SQL Data Cleaning, Transformation and Analysis: Raw Data of Open-Pit Copper Mine** <br><br>

*_**As there were no publicly available raw datasets for open-pit copper mine operations, synthetic data for an open-pit copper mine was generated using generative artificial intelligence.**_ <br>

*_**Nevertheless, it should be noted that all SQL data extraction, cleaning, transformation, and analysis presented here are entirely my own work.**_ <br><br><br>

The relational data model for the dataset is shown in the image below and consists of five tables, representing the following information:

1. **`01_energy_sources_raw`** – Energy sources used across the mine's operations <br><br>
   
2. **`02_operation_hierarchy_raw`** – The operations and sub-operations within the mine <br><br>
  
3. **`03_daily_production_raw`** – Daily mine production data <br><br>
   
4. **`04_daily_energy_consumption_raw`** – Daily energy consumption across the mine's operations <br><br>
  
5. **`05_monthly_energy_costs_raw`** – Monthly energy costs associated with the various energy sources used across different mine operations <br><br>
   
![Relational Data Model](Relational%20Data%20Model.png)

A data dictionary describing the columns contained within each table of the relational data model is provided below:

<img src="Data%20Dictionary.png" width="900">

The raw datasets contained inconsistent text formatting, mixed date formats, non-numeric values, missing values, different energy units, and values outside expected ranges. SQL was used to clean and transform the datasets before combining them for the final energy, cost and emissions analysis.<br><br><br><br>

## **Table 1 (01_energy_sources_raw) - Data Cleaning & Transformation:**
<br>

<img src="01_energy_sources_raw.png" width="1200">
<br>

A new table, **`energy_sources_clean`**, was created from the **`01_energy_sources_raw`** table using the `CREATE TABLE` statement. 

During this process, several data cleaning and transformation steps were applied:

* In the **`energy_source`** column, the `TRIM()` function was used to remove leading and trailing spaces, while the `UPPER()` function was applied to entirely capitalise the **`energy_source`** names, which initially had inconsistent capitalisations <br><br>

* In the **`standard_unit`** column, the energy units were standardised using a `CASE` statement to ensure consistency across the dataset. For example, values such as 'mwh' were converted to 'MWh', while 'litres' was standardised to 'L' <br><br>

* The **`emissions_factor_tco2e_per_unit`** column represents the emissions factor for each energy source, expressed as tonnes of CO₂e per unit of energy consumed. <br><br>As the original emissions factors were reported using different units depending on the energy source, i.e. MWh, GJ, and L, a new column named **`emissions_factor_tco2e_per_MJ`** was created in order to standardise the emissions factor values with respect to megajoules (MJ) <br><br>

* A new column, **`MJ_per_unit`**, was also created to store the conversion factor between each energy source's original unit and megajoules (MJ).<br><br>  The conversion factors used were: <br><br> 1 MWh = 3,6000 MJ <br><br> 1L = 40 MJ (Diesel) <br><br> 1 GJ = 1000 MJ <br><br> This conversion factor was then used to calculate the standardised values in the **`emissions_factor_tco2e_per_MJ`** column. <br><br> 

In [3]:
-- TABLE 1 - ENERGY SOURCES

--Create new table from original table 
CREATE TABLE energy_sources_clean AS 
SELECT * 
FROM '01_energy_sources_raw.csv';

--Standardise energy source units
UPDATE energy_sources_clean
SET energy_source = UPPER(TRIM(energy_source)),
	standard_unit = CASE WHEN standard_unit ILIKE '%mwh%' THEN 'MWh'
	                     WHEN standard_unit ILIKE '%litres%' THEN 'L'
	                	 ELSE standard_unit END;

--Add new column for storing conversion factor between energy source original unit and megajoules (MJ)
ALTER TABLE energy_sources_clean
ADD COLUMN MJ_per_unit NUMERIC;

--Input conversion factor values into new column 
UPDATE energy_sources_clean
SET MJ_per_unit = CASE WHEN standard_unit = 'MWh' THEN 3600
	                   WHEN standard_unit = 'L'   THEN 40
	                   WHEN standard_unit = 'GJ'  THEN 1000
	                   ELSE MJ_per_unit END;

--Add new column for storing emissions factor in tonnes CO2e per megajoule (MJ)
ALTER TABLE energy_sources_clean
ADD COLUMN emissions_factor_tco2e_per_MJ DOUBLE;

--Calculate emissions factor in tonnes CO2 per MJ, using conversion factor and original emissions factor 
UPDATE energy_sources_clean
SET emissions_factor_tco2e_per_MJ = emissions_factor_tco2e_per_unit/MJ_per_unit;

SELECT *
FROM energy_sources_clean;

,energy_source_id,energy_source,standard_unit,emissions_factor_tco2e_per_unit,MJ_per_unit,emissions_factor_tco2e_per_MJ
0,16706266,GRID ELECTRICITY,MWh,0.69,3600,0.0001916667
1,28450097,DIESEL,L,0.00268,40,0.000067
2,53385195,NATURAL GAS,GJ,0.0515,1000,0.0000515
3,22757633,SOLAR PPA,MWh,0.03,3600,0.0000083333


<br>

## **Table 2 (02_operation_hierarchy_raw) - Data Cleaning & Transformation:**
<br>

<img src="02_operation_hierarchy_raw.png" width="1000">
<br>
A new table, **`operation_hierarchy_clean`**, was created from the **`02_operation_hierarchy_raw`** table using the `CREATE TABLE` statement.

The data cleaning and transformation steps applied were as follows:

* The `TRIM()` function was used to remove leading and trailing spaces for both the **`main_operation`** and **`sub_operation`** columns, while the `UPPER()` function was applied to entirely capitalise the **`main_operation`** names, which initially had inconsistent capitalisations <br><br>

* One NULL value was identified in the **`energy_criticality`** column for the **`operational_area`** of 'Concentrate Handling'. Other records within the same operational area of 'Concentrate Handling' were assigned an **`energy_criticality`** of 'High'. <br><br> Therefore, the NULL value in the **`energy_criticality`** column was assigned 'High' as well



In [1]:
-- TABLE 2 - OPERATION HIERARCHY

--Create new table from original table 
CREATE TABLE operation_hierarchy_clean AS 
SELECT * 
FROM '02_operation_hierarchy_raw.csv';

--Trim and capitalise main_operation categories, and trim sub_operation categories
UPDATE operation_hierarchy_clean 
SET main_operation = TRIM(UPPER(main_operation)),
    sub_operation = TRIM(sub_operation);

--Fill in missing value in energy_criticality column 
UPDATE operation_hierarchy_clean
SET energy_criticality = 'High'
WHERE TRIM(energy_criticality) IS NULL;

SELECT *
FROM operation_hierarchy_clean
ORDER BY process_stage_order
LIMIT 10;

,operation_id,main_operation,sub_operation,process_stage_order,operational_area,energy_criticality,typical_schedule
0,91340540,MINE DEVELOPMENT,Grade-control drilling,1,Open pit - ore zones,Medium,24/7
1,24914615,MINE DEVELOPMENT,Blast-hole drilling,2,Open pit - active benches,High,24/7
2,62814762,MINE DEVELOPMENT,Blasting support,3,Open pit - active benches,Medium,Day shift
3,77640743,MINE DEVELOPMENT,Pit dewatering,4,Open pit sumps,High,24/7
4,83276760,MINE DEVELOPMENT,Haul-road maintenance,5,Pit and waste routes,Medium,24/7
5,61354486,MATERIAL MOVEMENT,Excavating and loading,6,Open pit loading faces,High,24/7
6,49569161,MATERIAL MOVEMENT,Ore hauling,7,Pit to ROM pad,High,24/7
7,98042943,MATERIAL MOVEMENT,Waste hauling,8,Pit to waste dumps,High,24/7
8,34819938,MATERIAL MOVEMENT,Dozing and grading,9,Pit and stockpiles,Medium,24/7
9,98788663,MATERIAL MOVEMENT,Stockpile Reclaim,10,ROM stockpile,High,24/7


<br>

## **Table 3 (03_daily_production_raw) - Data Cleaning & Transformation:**

<br>

<img src="03_daily_production_raw.png" width="1000">
<br>
A new table, **`daily_production_clean`**, was created from the **`03_daily_production_raw`** table using the `CREATE TABLE` statement.

The data cleaning and transformation steps applied were as follows:

* The **`production_date`** column was converted to the DATE data type. As the dates were recorded in several different formats, the `TRY_STRPTIME()` function (a function from the DuckDB database management system) was used to parse the values by specifying each expected date format.<br><br>`TRY_STRPTIME()` initially converted the values to TIMESTAMP, after which they were converted to DATE using the `CAST` shorthand of `::`<br><br>

* The `TRIM()` function was used to remove leading and trailing spaces of the entries in the **`metric_name`** column, while the `UPPER()` function was applied to entirely capitalise the entries, which initially had inconsistent capitalisations <br><br>

* The **`unit`** column contained inconsistent unit labels, including values such as 't', 'tonnes', '% Cu', 'pct Cu', '%', and 'percent'. These labels were standardised to either 't' or '%' using a `CASE` statement <br><br> 

* The **`metric_value`** column was initially stored as `VARCHAR` and was converted to the `NUMERIC` data type using `CAST()`. Prior to conversion, the values were cleaned to prevent data type conversion errors. The `TRIM()` function was used to remove leading and trailing spaces, `REPLACE()` was used to remove commas and replace it with an empty string, and `NULLIF()` was applied to convert blank entries to `NULL`<br><br> 

* Some values in the **`metric_value`** column were recorded as negative, even though negative values were not valid for those measurements. These were treated as likely data-entry errors and corrected using the `ABS()` function<br><br> 

* The **`metric_value`** column also contained several outlier values that substantially deviated from the typical range observed under a given **`metric_name`**.<br><br> Based on the surrounding values under the same **`metric_name`**, these outlier values appeared to be decimal-place entry errors.<br><br> The outlier values in **`metric_value`** were therefore divided by 10 to bring them in line with the expected range for a given **`metric_name`**. <br><br> This was carried out using a `CASE` statement addressing each **`metric_name`**<br><br> 

In [2]:
-- TABLE 3 - DAILY PRODUCTION 

--Create new table from original table
CREATE TABLE daily_production_clean AS 
SELECT *
FROM '03_daily_production_raw.csv';

--Convert production_date column from VARCHAR to DATE type, taking into account the various date formats when converting
ALTER TABLE daily_production_clean
ALTER COLUMN production_date TYPE DATE
USING TRY_STRPTIME(
    TRIM(production_date),
    ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d','%m-%d-%Y','%m/%d/%Y','%d-%B-%Y','%d-%b-%Y']
	)::DATE;

--Trim and capitalise metric_name column, standardise unit column 
UPDATE daily_production_clean
SET metric_name = TRIM(UPPER(metric_name)),
	unit = CASE WHEN unit ILIKE '%tonnes%' THEN 't'
	            WHEN unit ILIKE '%Cu%' THEN '%'
	            WHEN unit ILIKE '%p%' THEN '%'
	            ELSE unit END; 

--Clean metric_value column, followed by converting metric_value column from VARCHAR to NUMERIC type 
ALTER TABLE daily_production_clean
ALTER COLUMN metric_value TYPE NUMERIC
USING CAST(NULLIF(REPLACE(TRIM(metric_value),',',''),'') AS NUMERIC);

--Convert negative metric_value entries to positive. Adjusting decimal place error values in metric_value column, with respect to each metric_name
UPDATE daily_production_clean
SET metric_value = CASE WHEN metric_value < 0 THEN ABS(metric_value)
	                    WHEN metric_name = 'COPPER CONCENTRATE PRODUCED' AND metric_value > 1000 THEN metric_value/10
	                    WHEN metric_name = 'COPPER RECOVERY' AND metric_value > 100 THEN metric_value/10
	                    WHEN metric_name = 'ORE PROCESSED' AND metric_value > 40000 THEN metric_value/10
	                    ELSE metric_value END;

SELECT *
FROM daily_production_clean
ORDER BY production_date
LIMIT 10;

,production_record_id,production_date,operation_id,metric_name,metric_value,unit,shift_reporting
0,11535043,2025-01-01,98042943,TOTAL MATERIAL MINED,107328.412,t,Daily total
1,77562948,2025-01-01,49569161,ORE MINED,26985.474,t,Daily total
2,96003108,2025-01-01,58956654,ORE PROCESSED,29480.207,t,Daily total
3,90389400,2025-01-01,15289718,COPPER CONCENTRATE PRODUCED,733.293,t,Daily total
4,41973496,2025-01-01,45614007,HEAD GRADE,0.723,%,Daily total
5,66958569,2025-01-01,45769449,COPPER RECOVERY,87.705,%,Daily total
6,67578889,2025-01-01,73642032,RECLAIM WATER RECOVERY,69.060,%,Daily total
7,42471207,2025-01-02,98042943,TOTAL MATERIAL MINED,107589.906,t,Day shift
8,30191546,2025-01-02,49569161,ORE MINED,26872.621,t,Daily total
9,75517762,2025-01-02,58956654,ORE PROCESSED,25351.793,t,Daily total


<br>

## **Table 4 (04_daily_energy_consumption_raw) - Data Cleaning & Transformation:**

<br>

<img src="04_daily_energy_consumption_raw.png" width="1000">
<br>
A new table, **`daily_energy_consumption_clean`**, was created from the **`04_daily_energy_consumption_raw`** table using the `CREATE TABLE` statement.

The data cleaning and transformation steps applied were as follows:

* The **`consumption_date`** column was converted to the DATE data type. As the dates were recorded in several different formats, the `TRY_STRPTIME()` function (a function from the DuckDB database management system) was used to parse the values by specifying each expected date format.<br><br>`TRY_STRPTIME()` initially converted the values to TIMESTAMP, after which they were converted to DATE using the `CAST` shorthand of `::`<br><br>

* The **`consumption_quantity`** column contained several data quality issues that needed to be addressed before it could be converted to the `NUMERIC` data type. A `CASE` statement was used to apply the required cleaning steps.<br><br> First, values that were blank after applying `TRIM()` were converted to `NULL`. <br><br> Next, entries containing alphabetic characters were also converted to `NULL` using the `REGEXP_MATCHES()` function; in this dataset, many of these entries were recorded as 'N/A'. <br><br> Finally, commas were then removed from numeric values using the `REPLACE()` function.<br><br> These cleaning steps helped prevent errors during the subsequent conversion of the **`consumption_quantity`** column to `NUMERIC`. <br><br> After conversion, any negative values were assumed to be likely due to data-entry errors, and converted to positive values using the `ABS()` function<br><br>

* In the **`reported_unit`** column, energy unit labels were standardised using a `CASE` statement to ensure consistency across the dataset. For example, values such as 'mwh' were converted to 'MWh', while 'l' was standardised to 'L'<br><br>

* The rows in the **`daily_energy_consumption_clean`** table were matched with that of the **`operation_hierarchy_clean`** table, through the **`operation_id`**. <br><br> The **`consumption_quantity`** contained several outlier values that substantially deviated from the typical range observed under a given **`sub_operation`**. <br><br> Based on the surrounding values under the same **`sub_operation`**, these outlier values appeared to be decimal-place entry errors.<br><br> The outlier values in **`consumption_quantity`** were therefore divided by 10 to bring them in line with the expected range for a given **`sub_operation`**.<br><br> This was carried out using a `CASE` statement addressing each **`sub_operation`**<br><br>

* A new column, **`consumption_quantity_MJ`**, was created to standardise energy consumption values into megajoules (MJ). <br><br> The original **`consumption_quantity`** values were recorded in different units, such as MWh, GJ, and L. These values were converted to MJ using the corresponding **`MJ_per_unit`** conversion factor from the previously created **`energy_sources_clean`** table. <br><br> The **`MJ_per_unit`** was matched to the correct **`consumption_quantity`** through the **`energy_source_id`**.<br><br> Therefore, **`consumption_quantity_MJ`** was calculated using the formula: <br><br> **`consumption_quantity_MJ`** = **`consumption_quantity`** x **`MJ_per_unit`**<br><br>

In [1]:
-- TABLE 4 - DAILY ENERGY CONSUMPTION

--Create new table from original table
CREATE TABLE daily_energy_consumption_clean AS 
SELECT *
FROM '04_daily_energy_consumption_raw.csv';

--Convert consumption_date column from VARCHAR to DATE type, taking into account the various date formats when converting
ALTER TABLE daily_energy_consumption_clean
ALTER COLUMN consumption_date TYPE DATE
USING TRY_STRPTIME(
    TRIM(consumption_date),
    ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d','%m-%d-%Y','%m/%d/%Y','%d-%B-%Y','%d-%b-%Y']
	)::DATE;

--Trim, return NULL value for rows containing alphabets, and remove commas in consumption_quantity column. Standardise reported_unit column
UPDATE daily_energy_consumption_clean 
SET consumption_quantity = CASE WHEN TRIM(consumption_quantity) LIKE '' THEN NULL 
	                            WHEN REGEXP_MATCHES(TRIM(consumption_quantity),'[a-zA-Z]') THEN NULL
	                            ELSE REPLACE(TRIM(consumption_quantity),',','') END,
	            
	
	reported_unit = CASE WHEN reported_unit ILIKE '%mwh%' THEN 'MWh'
	                     WHEN reported_unit ILIKE '%l%' THEN 'L'
	                     ELSE TRIM(UPPER(reported_unit)) END;

--Convert consumption_quantity column from VARCHAR to NUMERIC type	                	
ALTER TABLE daily_energy_consumption_clean 
ALTER COLUMN consumption_quantity TYPE NUMERIC
USING CAST (consumption_quantity AS NUMERIC);

--Convert negative consumption_quantity entries to positive. Adjusting decimal place error values in consumption_quantity column, with respect to each sub_operation
UPDATE daily_energy_consumption_clean AS d
SET consumption_quantity = CASE WHEN consumption_quantity < 0 THEN ABS(consumption_quantity)
	                            WHEN sub_operation = 'Blast-hole drilling' AND consumption_quantity > 15000 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Ore hauling' AND consumption_quantity > 50000 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Waste hauling' AND consumption_quantity > 70000 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Secondary crushing' AND consumption_quantity > 150 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Tailings pumping' AND consumption_quantity > 200 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Reclaim-water pumping' AND consumption_quantity > 100 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Administration and camp' AND consumption_quantity > 60 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Pit dewatering' AND consumption_quantity > 50 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Haul-road maintenance' AND consumption_quantity > 9000 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Stockpile Reclaim' AND consumption_quantity > 4000 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Ball milling' AND consumption_quantity > 500 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Concentrate thickening' AND consumption_quantity > 100 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Concentrate filtration' AND consumption_quantity > 100 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Concentrate drying' AND consumption_quantity > 200 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Tailings thickening' AND consumption_quantity > 100 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Workshop maintenance' AND consumption_quantity > 3000 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Lighting and communications' AND consumption_quantity > 30 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Laboratory' AND consumption_quantity > 20 THEN consumption_quantity/10
	                            WHEN sub_operation = 'Dozing and grading' AND consumption_quantity > 20000 THEN consumption_quantity/10
	                            ELSE consumption_quantity END
FROM operation_hierarchy_clean AS o 
WHERE d.operation_id = o.operation_id;

--Add new column for storing energy consumption quantity in megajoules (MJ)
ALTER TABLE daily_energy_consumption_clean 
ADD COLUMN consumption_quantity_MJ NUMERIC;

--Convert consumption_quantity into consumption_quantity_MJ using the energy conversion factor, MJ_per_unit
UPDATE daily_energy_consumption_clean AS d
SET consumption_quantity_MJ = consumption_quantity*e.MJ_per_unit
FROM energy_sources_clean AS e
WHERE e.energy_source_id = d.energy_source_id;

SELECT *
FROM daily_energy_consumption_clean
ORDER BY consumption_date
LIMIT 10;

,energy_record_id,consumption_date,operation_id,energy_source_id,consumption_quantity,reported_unit,runtime_hours,peak_demand_kw,consumption_quantity_MJ
0,99854650,2025-01-01,91340540,28450097,3936.104,L,21.07,NaN,157444.16
1,10309729,2025-01-01,24914615,28450097,10249.210,L,24.00,NaN,409968.40
2,95810819,2025-01-01,62814762,28450097,2419.749,L,17.79,NaN,96789.96
3,13654276,2025-01-01,77640743,16706266,33.085,MWh,22.05,1771.5,119106.00
4,71256835,2025-01-01,77640743,22757633,2.022,MWh,23.87,112.1,7279.20
5,50340543,2025-01-01,83276760,28450097,6003.163,L,22.19,NaN,240126.52
6,23141757,2025-01-01,61354486,28450097,21093.560,L,24.00,NaN,843742.40
7,70222208,2025-01-01,49569161,28450097,32284.320,L,20.05,NaN,1291372.80
8,61630856,2025-01-01,98042943,28450097,55959.909,L,22.35,NaN,2238396.36
9,97082984,2025-01-01,34819938,28450097,12202.121,L,22.24,NaN,488084.84


<br>

## **Table 5 (05_monthly_energy_costs_raw) - Data Cleaning & Transformation:**

<br>

<img src="05_monthly_energy_costs_raw.png" width="1000">
<br>
A new table, **`monthly_energy_costs_clean`**, was created from the **`05_monthly_energy_costs_raw`** table using the `CREATE TABLE` statement.

The data cleaning and transformation steps applied were as follows:

* The **`billing_month`** column was converted to the DATE data type. As the dates were recorded in several different formats, the `TRY_STRPTIME()` function (a function from the DuckDB database management system) was used to parse the values by specifying each expected date format.<br><br>`TRY_STRPTIME()` initially converted the values to TIMESTAMP, after which they were converted to DATE using the `CAST` shorthand of `::`<br><br>

* The **`unit_cost_aud`** column was initially stored as `VARCHAR` and needed to be converted to the `NUMERIC` data type. Before conversion, the values were cleaned to prevent data type conversion errors.<br><br> Some entries in the **`unit_cost_aud`** column contained the text 'AUD'. This was removed using the `REPLACE()` function by replacing 'AUD' with an empty string. The `TRIM()` function was then applied to remove any leading or trailing spaces before converting the cleaned values to `NUMERIC` <br><br>

* In the **`billed_unit`** column, energy unit labels were standardised using a `CASE` statement to ensure consistency across the dataset. Firstly, values such as 'mwh' were converted to 'MWh'. For the remaining entries, `TRIM()` was used to remove leading and trailing spaces, while `UPPER()` was applied to capitalise the unit labels, i.e. 'L' and 'GJ' <br><br>

* A new column, **`cost_per_MJ`**, was created to standardise energy costs to Australian dollars per megajoule (AUD/MJ), instead of expressing costs in terms of MWh, GJ, or L in the **`unit_cost_aud`** column. <br><br> The values in **`unit_cost_aud`** were converted to **`cost_per_MJ`** using the corresponding **`MJ_per_unit`** conversion factor from the previously created **`energy_sources_clean`** table.<br><br> The appropriate conversion factor was matched to each record using the **`energy_source_id`**. <br><br> Hence, the **`cost_per_MJ`** values were calculated using the following formula: <br><br> **`cost_per_MJ = unit_cost_aud / MJ_per_unit`**<br><br>

* In the **`supplier`** column, the supplier names were standardised using a `CASE` statement to ensure consistency across the dataset. Firstly, the supplier name 'Sun West PPA' and 'SunWest PPA' were standardised to 'SUNWEST PPA'. For the remaining entires, `TRIM()` was used to remove leading and trailing spaces, while the `UPPER()` function was applied to entirely capitalise the **`supplier`** names, which initially had inconsistent capitalisations <br><br>

In [1]:
-- TABLE 5 - MONTHLY ENERGY COSTS

--Create new table from original table
CREATE TABLE monthly_energy_costs_clean AS 
SELECT *
FROM '05_monthly_energy_costs_raw.csv';

--Convert billing_month column from VARCHAR to DATE type, taking into account the various date formats when converting
ALTER TABLE monthly_energy_costs_clean
ALTER COLUMN billing_month TYPE DATE
USING TRY_STRPTIME(
    TRIM(billing_month),
    ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d','%m-%d-%Y','%m/%d/%Y','%m/%Y','%m-%Y','%Y/%m','%Y-%m','%B-%Y','%b-%Y']
)::DATE; 

--Replace alphabets from unit_cost_aud column with empty strings
UPDATE monthly_energy_costs_clean 
SET unit_cost_aud = TRIM(REPLACE(unit_cost_aud,'AUD',''))
WHERE unit_cost_aud LIKE '%AUD%';

--Convert unit_cost_aud column from VARCHAR to NUMERIC
ALTER TABLE monthly_energy_costs_clean
ALTER COLUMN unit_cost_aud TYPE NUMERIC
USING CAST (unit_cost_aud AS NUMERIC);

--Standardise billed_unit column 
UPDATE monthly_energy_costs_clean 
SET billed_unit = CASE WHEN billed_unit ILIKE '%mwh%' THEN 'MWh'
	                   ELSE TRIM(UPPER(billed_unit)) END;

--Add new column for storing energy cost per megajoule (MJ)
ALTER TABLE monthly_energy_costs_clean
ADD COLUMN cost_per_MJ DOUBLE; 

--Convert unit_cost_aud into cost_per_MJ using the energy conversion factor, MJ_per_unit
UPDATE monthly_energy_costs_clean AS m
SET cost_per_MJ =  unit_cost_aud/e.MJ_per_unit,
	supplier = CASE WHEN supplier ILIKE '%Sun%' THEN 'SUNWEST PPA'
		            ELSE TRIM(UPPER(supplier)) END
FROM energy_sources_clean AS e
WHERE e.energy_source_id = m.energy_source_id;

SELECT *
FROM monthly_energy_costs_clean
ORDER BY billing_month
LIMIT 10;

,cost_record_id,billing_month,energy_source_id,unit_cost_aud,billed_unit,supplier,cost_per_MJ
0,83578961,2025-01-01,16706266,144.052,MWh,INLAND GRID RETAIL,0.040014
1,40058615,2025-01-01,28450097,1.898,L,SOUTHERN FUELS,0.047450
2,62582859,2025-01-01,53385195,13.116,GJ,REGIONAL GAS SUPPLY,0.013116
3,37985542,2025-01-01,22757633,85.744,MWh,SUNWEST PPA,0.023818
4,14002662,2025-02-01,16706266,150.207,MWh,INLAND GRID RETAIL,0.041724
5,36317967,2025-02-01,28450097,1.894,L,SOUTHERN FUELS,0.047350
6,44772181,2025-02-01,53385195,12.317,GJ,REGIONAL GAS SUPPLY,0.012317
7,97436300,2025-02-01,22757633,86.257,MWh,SUNWEST PPA,0.023960
8,28614326,2025-03-01,16706266,155.861,MWh,INLAND GRID RETAIL,0.043295
9,26637712,2025-03-01,28450097,1.863,L,SOUTHERN FUELS,0.046575


<br>

## **DATA ANALYSIS: Energy, Cost and Emissions Analysis of an Open-Pit Copper Mine**

Following the completion of the data cleaning and transformation, the cleaned tables were used to calculate the mine's energy consumption, energy costs, and greenhouse gas emissions. 

The energy consumption, energy costs, and greenhouse gas emissions were calculated on a per tonne of copper concentrate basis, by dividing the total values for each parameter by the total amount of copper concentrate produced.

The data analysis steps applied were as follows:

1. A Common Table Expression (CTE) named **`cost`** was first created to calculate the energy cost associated for each energy consumption record. The **`monthly_energy_costs_clean`** table was joined with the **`daily_energy_consumption_clean`** table using the **`energy_source_id`**. <br><br> The month of each **`consumption_date`** was matched to the corresponding month in **`billing_month`** using the `EXTRACT(MONTH FROM ...)` function. This ensured that the appropriate monthly energy cost was applied to each daily energy consumption record. <br><br> The total **`energy_cost`** for each record was then calculated by multiplying the energy consumed in megajoules, **`consumption_quantity_MJ`**, by the corresponding **`cost_per_MJ`**: <br><br> **`energy_cost = consumption_quantity_MJ × cost_per_MJ`** <br><br>

2. A second CTE, **`co2emissions_energyconsumption_energycost`**, was then created to combine the energy consumption, greenhouse gas emissions, and energy cost data with the operational hierarchy and energy source information. <br><br> The **`operation_hierarchy_clean`**, **`daily_energy_consumption_clean`**, **`energy_sources_clean`**, and previously created **`cost`** CTE were joined using their corresponding **`operation_id`** and **`energy_source_id`** fields.<br><br> The aggregate function of `SUM()` was then used to calculate the total energy consumption, total greenhouse gas emissions, and total energy cost.<br><br> The aggregation was grouped by **`main_operation`**, **`sub_operation`**, and **`energy_source`**. <br><br> Total energy consumption was calculated by summing the values in **`consumption_quantity_MJ`**, while total energy cost was determined by summing the values in **`energy_cost`**.<br><br> Total greenhouse gas emissions were calculated by multiplying **`emissions_factor_tco2e_per_MJ`**, with the **`consumption_quantity_MJ`**, and then summing these values <br><br>

3. The final query was used to calculate the energy consumption, greenhouse gas emissions, and energy costs with respect to per tonne of copper concentrate produced.<br><br> **It is important to note that the total greenhouse gas emissions (CO2e) in tonnes was converted to kg by multiplying by 1000, before calculating it with respect to per tonne of copper concentrate. <br><br> (The total copper concentrate produced was obtained from the **`daily_production_clean`** table by summing **`metric_value`** where **`metric_name = 'COPPER CONCENTRATE PRODUCED'`**. This calculation was nested in a subquery.) <br><br> The final results were rounded to 4-5 decimal places, and ordered according to **`main_operation`**, **`sub_operation`**, and **`energy_source`**

In [3]:
--DATA ANALYSIS

--Common Table Expression (CTE) for calculating energy cost for each energy consumption record
WITH cost AS 
	(SELECT  
	     d.operation_id,
	     d.energy_source_id,
	 	(d.consumption_quantity_MJ * m.cost_per_MJ) AS energy_cost
	
	 FROM monthly_energy_costs_clean AS m
	
	 INNER JOIN daily_energy_consumption_clean AS d
	 ON d.energy_source_id = m.energy_source_id
	
	 WHERE EXTRACT(MONTH FROM d.consumption_date) = EXTRACT(MONTH FROM m.billing_month)
	),

--Common Table Expression (CTE) for calculating total energy consumption, total greenhouse gas emissions and total energy costs
co2emissions_energyconsumption_energycost AS 
	(SELECT 
	   o.main_operation, 
	   o.sub_operation,
	   e.energy_source,
	   SUM(d.consumption_quantity_MJ) AS total_consumption_MJ,
	   SUM(e.emissions_factor_tco2e_per_MJ * d.consumption_quantity_MJ) AS total_tco2e_emissions,
	   SUM(c.energy_cost) AS total_energy_cost
	
	FROM operation_hierarchy_clean AS o
	
	INNER JOIN daily_energy_consumption_clean AS d
	ON o.operation_id = d.operation_id
	
	INNER JOIN energy_sources_clean AS e
	ON e.energy_source_id = d.energy_source_id
	
	INNER JOIN cost AS c
	ON c.operation_id = o.operation_id
	AND c.energy_source_id = d.energy_source_id
	
	GROUP BY o.main_operation, o.sub_operation, e.energy_source
	ORDER BY o.main_operation, o.sub_operation, e.energy_source
	)

--Main query referencing the co2emissions_energyconsumption_energycost CTE
--Subquery calculating the total tonnes of copper concentrate produced
--Calculation of energy consumption, energy cost, and greenhouse gas emissions per tonne of copper concentrate
--Round final results to 4-5 decimal places. Order results by main_operation, sub_operation and energy_source
SELECT main_operation, 
	
	   sub_operation, 

	   energy_source,
	
	   ROUND(total_consumption_MJ/(SELECT SUM(metric_value) FROM daily_production_clean WHERE metric_name = 'COPPER CONCENTRATE PRODUCED'),4) AS MJ_consumption_per_tonne_concentrate,
	
	   ROUND((total_tco2e_emissions*1000)/(SELECT SUM(metric_value) FROM daily_production_clean WHERE metric_name = 'COPPER CONCENTRATE PRODUCED'),4) AS kgco2e_emissions_per_tonne_concentrate,

	   ROUND(total_energy_cost/(SELECT SUM(metric_value) FROM daily_production_clean WHERE metric_name = 'COPPER CONCENTRATE PRODUCED'),5) AS energy_cost_per_tonne_concentrate
	
FROM co2emissions_energyconsumption_energycost
	
ORDER BY main_operation, sub_operation, energy_source
	
LIMIT 10;

,main_operation,sub_operation,energy_source,MJ_consumption_per_tonne_concentrate,kgco2e_emissions_per_tonne_concentrate,energy_cost_per_tonne_concentrate
0,MATERIAL MOVEMENT,Dozing and grading,DIESEL,186.2206,12.4768,0.82586
1,MATERIAL MOVEMENT,Excavating and loading,DIESEL,307.4003,20.5958,1.36447
2,MATERIAL MOVEMENT,Ore hauling,DIESEL,619.2368,41.4889,2.73793
3,MATERIAL MOVEMENT,Stockpile Reclaim,DIESEL,47.6009,3.1893,0.21128
4,MATERIAL MOVEMENT,Stockpile Reclaim,GRID ELECTRICITY,53.3513,10.2257,0.22773
5,MATERIAL MOVEMENT,Stockpile Reclaim,SOLAR PPA,8.6736,0.0723,0.02091
6,MATERIAL MOVEMENT,Waste hauling,DIESEL,924.8382,61.9642,4.10493
7,MINE DEVELOPMENT,Blast-hole drilling,DIESEL,247.3561,16.5729,1.09727
8,MINE DEVELOPMENT,Blasting support,DIESEL,55.2691,3.7030,0.24537
9,MINE DEVELOPMENT,Grade-control drilling,DIESEL,99.3839,6.6587,0.44110
